In [191]:
import duckdb, boto3, os, requests, tarfile, gdown, gzip, time
import polars as pl
from pathlib import Path
import xml.etree.ElementTree as ET
os.makedirs('data', exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/xmls", exist_ok=True)
os.makedirs("data/parquet", exist_ok=True)

**Please note**: the below link is a place-holder... you need to request access from HIVDB if you want the TCE dataset!

In [25]:
url = "https://drive.google.com/file/d/19PRBBOEFZGalIXE8am64L1e6dyBlF5K_/view?usp=sharing"
tar_path = "data/raw/tce_xmls.tar.gz"

In [24]:
try:
    gdown.download(url, tar_path, quiet=False)
except Exception as e:
    print(f"Download failed: {e}")
    print("Please request the dataset from Stanford HIVDB directly")

Downloading...
From: https://drive.google.com/uc?id=19PRBBOEFZGalIXE8am64L1e6dyBlF5K_
To: /Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion/data/raw/tce_xmls.tar.gz
100%|██████████| 24.8M/24.8M [00:04<00:00, 4.96MB/s]


In [22]:
import tarfile

with tarfile.open("data/raw/tce_xmls.tar.gz", "r:") as tar:
    tar.extractall(path="data/xmls", filter='tar')

---

Parse the XML to Parquet.

In [215]:
def parse_tce_regimen_rows(xml_path):
    root = ET.parse(xml_path).getroot()

    patient = root.findtext('patient/patientAlias')
    filename = Path(xml_path).name

    rows = []

    # baseline
    for i, tx in enumerate(root.findall(".//baselineNewTreatment"), 1):
        start = tx.findtext("relativeStartDate")
        stop = tx.findtext("relativeStopDate")
        duration = tx.findtext("duration")

        for drug in tx.findall("drug"):
            rows.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "regimen_type": "baseline",
                "regimen_index": i,
                "start": start,
                "stop": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
            })

    # past
    for i, tx in enumerate(root.findall(".//pastRegimenTreatments"), 1):
        start = tx.findtext("relativeStartDate")
        stop = tx.findtext("relativeStopDate")
        duration = tx.findtext("duration")

        for drug in tx.findall("drug"):
            rows.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "regimen_type": "past",
                "regimen_index": i,
                "start": start,
                "stop": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
            })

    return rows


def process_all_xml(xml_dir):
    xml_files = list(Path(xml_dir).glob("*.xml"))
    all_rows = []

    for f in xml_files:
        try:
            all_rows.extend(parse_tce_regimen_rows(f))
        except Exception as e:
            print(f"Failed {f.name}: {e}")

    return pl.DataFrame(all_rows)


df = process_all_xml("data/xmls/xmls_db")

In [216]:
for key, subdf in df.partition_by("regimen_type", as_dict=True).items():
    regimen_val = key[0] if isinstance(key, tuple) else key
    subdf = subdf.drop("regimen_type")
    subdf.write_parquet(
        f"data/parquet/tce_regimens/regimen_type={regimen_val}/data.parquet",
        mkdir=True
    )

In [217]:
df.head()

xml_filename,patient_alias,regimen_type,regimen_index,start,stop,duration,drug_code,drug_class
str,str,str,i64,str,str,str,str,str
"""162.xml""","""4303""","""baseline""",1,"""0""","""44""","""43""","""ABC""","""NRTI"""
"""162.xml""","""4303""","""baseline""",1,"""0""","""44""","""43""","""D4T""","""NRTI"""
"""162.xml""","""4303""","""baseline""",1,"""0""","""44""","""43""","""3TC""","""NRTI"""
"""162.xml""","""4303""","""past""",1,"""-28""","""0""","""27""","""ABC""","""NRTI"""
"""162.xml""","""4303""","""past""",1,"""-28""","""0""","""27""","""EFV""","""NNRTI"""


In [218]:
print(os.listdir("data/parquet/tce_regimens/"))

['regimen_type=baseline', 'regimen_type=past']


---

Upload to S3.

In [219]:
def upload_to_s3(local_dir, bucket, prefix, s3_client):
    """Upload partitioned Parquet to S3."""
    local = Path(local_dir)
    for file in local.rglob('*.parquet'):
        key = f"{prefix}{file.relative_to(local)}"
        s3_client.upload_file(str(file), bucket, key)
        print(f"Uploaded: s3://{bucket}/{key}")

# Upload the partitioned datasets.
upload_to_s3('data/parquet/tce_regimens/', bucket_name, 'hivdb-tce/tce_regimens/', s3)

Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_regimens/regimen_type=baseline/data.parquet
Uploaded: s3://hiv-data-022784797781/hivdb-tce/tce_regimens/regimen_type=past/data.parquet


---

Use Glue to crawl the data.

In [199]:
def create_crawler(name, s3_path, database='hivdb_tce'):
    """Create Glue crawler for partitioned Parquet data."""
    glue = session.client('glue')
    
    # Get your account ID
    sts = session.client('sts')
    account_id = sts.get_caller_identity()['Account']
    
    # Use your account ID in the role ARN
    role_arn = f'arn:aws:iam::{account_id}:role/glue_crawler_role'
    
    glue.create_crawler(
        Name=name,
        Role=role_arn,
        DatabaseName=database,
        Targets={'S3Targets': [{'Path': s3_path}]},
        SchemaChangePolicy={
            'UpdateBehavior': 'UPDATE_IN_DATABASE',
            'DeleteBehavior': 'DEPRECATE_IN_DATABASE'
        }
    )
create_crawler('tce_regimens_crawler', f's3://{bucket_name}/hivdb-tce/tce_regimens/')

In [220]:
glue = session.client('glue')

glue.get_crawler(Name='tce_regimens_crawler')

{'Crawler': {'Name': 'tce_regimens_crawler',
  'Role': 'glue_crawler_role',
  'Targets': {'S3Targets': [{'Path': 's3://hiv-data-022784797781/hivdb-tce/tce_regimens/',
     'Exclusions': []}],
   'JdbcTargets': [],
   'MongoDBTargets': [],
   'DynamoDBTargets': [],
   'CatalogTargets': [],
   'DeltaTargets': [],
   'IcebergTargets': [],
   'HudiTargets': []},
  'DatabaseName': 'hivdb_tce',
  'Classifiers': [],
  'RecrawlPolicy': {'RecrawlBehavior': 'CRAWL_EVERYTHING'},
  'SchemaChangePolicy': {'UpdateBehavior': 'UPDATE_IN_DATABASE',
   'DeleteBehavior': 'DEPRECATE_IN_DATABASE'},
  'LineageConfiguration': {'CrawlerLineageSettings': 'DISABLE'},
  'State': 'READY',
  'CrawlElapsedTime': 0,
  'CreationTime': datetime.datetime(2026, 4, 15, 22, 9, 23, tzinfo=tzlocal()),
  'LastUpdated': datetime.datetime(2026, 4, 15, 22, 9, 23, tzinfo=tzlocal()),
  'LastCrawl': {'Status': 'SUCCEEDED',
   'LogGroup': '/aws-glue/crawlers',
   'LogStream': 'tce_regimens_crawler',
   'MessagePrefix': '02b94251-ec

In [221]:
glue.start_crawler(Name='tce_regimens_crawler')

{'ResponseMetadata': {'RequestId': '61f851e1-d8df-415d-839e-ab0d6ef66d70',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 02:42:50 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '61f851e1-d8df-415d-839e-ab0d6ef66d70',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [260]:
glue.get_crawler(Name='tce_regimens_crawler')['Crawler']['State']

'READY'

In [262]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='tce_regimens'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'regimen_index',
 'start',
 'stop',
 'duration',
 'drug_code',
 'drug_class']

---

Athena query checking.

In [192]:
athena = boto3.client("athena", region_name="us-east-1")

In [193]:
def run_athena_query(sql, database="hivdb_tce", output="s3://hiv-data-022784797781/athena-results/"):
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output},
    )
    qid = resp["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status["QueryExecution"]["Status"]["State"]

        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown error")
            raise RuntimeError(f"Athena query {state}: {reason}")

        time.sleep(1)

    results = athena.get_query_results(QueryExecutionId=qid)
    rows = results["ResultSet"]["Rows"]

    headers = [c["VarCharValue"] for c in rows[0]["Data"]]
    data = []
    for row in rows[1:]:
        values = [col.get("VarCharValue") for col in row.get("Data", [])]
        data.append(dict(zip(headers, values)))

    return data

rows = run_athena_query("""
    SELECT year_partition, COUNT(*) AS n
    FROM tce_by_year
    GROUP BY year_partition
    ORDER BY year_partition
""")

print(rows[:5])

[{'year_partition': '1997', 'n': '2'}, {'year_partition': '1998', 'n': '91'}, {'year_partition': '1999', 'n': '246'}, {'year_partition': '2000', 'n': '216'}, {'year_partition': '2001', 'n': '192'}]
